# 01 · Data Preparation

Audit et nettoyage des sept tables publiques IMDb. Les données brutes restent dans `data/raw` et une préparation terminée est publiée en Parquet. Les profils couvrent toutes les colonnes, avant et après nettoyage.

[Schéma IMDb](https://data.imdb.com/non-commercial-datasets/)

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / "src/imdb_loader.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Lancez le notebook depuis le dépôt IMDb ou son dossier notebooks.")
sys.path.insert(0, str(ROOT))

from pyspark.sql import functions as F
from src.data_quality import prepare
from src.imdb_loader import data_root, spark_session

DATA = data_root(ROOT)
spark = spark_session("IMDb Data Preparation")

## Q1 · Charger et mesurer les valeurs manquantes

Les chaînes vides, les chaînes composées d'espaces et le marqueur IMDb sont comptés avec les valeurs nulles. Le taux utilise toutes les lignes de la table, y compris celles dont la clé est invalide.

In [ ]:
snapshot = prepare(spark, DATA, expand_gzip=os.getenv("IMDB_EXPAND_GZIP", "1") != "0")
print(f"Préparation publiée : {snapshot}")

missing = spark.read.parquet((snapshot / "quality/missing").as_uri())
raw_profile = missing.filter(F.col("stage") == "raw")
raw_profile.orderBy("table", "column").drop("stage").withColumn(
    "missing_pct", F.round("missing_pct", 3)
).show(100, truncate=False)

## Q2 · Cinq colonnes les plus incomplètes

Le classement compare les pourcentages bruts de l'ensemble des tables. Les égalités sont départagées par le nom de table puis celui de colonne.

In [ ]:
top_missing = raw_profile.drop("stage").orderBy(
    F.desc_nulls_last("missing_pct"), "table", "column"
).limit(5)
top_missing.show(truncate=False)

## Q3 · Anomalies et exemples

Les contrôles couvrent les types et bornes numériques, les années inversées, les durées de vie suspectes, les clés invalides, les doublons et les références orphelines. Une même ligne peut relever de plusieurs contrôles. Les références orphelines sont auditées sur les clés exploitables après nettoyage structurel.

In [ ]:
anomalies = spark.read.parquet((snapshot / "quality/anomalies").as_uri())
anomalies.select("table", "issue", "count", "unit", "action").orderBy(
    F.desc("count"), "table", "issue"
).show(100, truncate=False)
anomalies.filter(F.col("count") > 0).select("table", "issue", "example").orderBy(
    "table", "issue"
).show(100, truncate=160)

## Q4 · Traitement et contrôle avant/après

Les champs numériques invalides sont mis à null. Les doublons identiques après normalisation sont regroupés. Toutes les variantes d'une clé conflictuelle sont écartées et l'audit conserve deux variantes représentatives. Une durée de vie suspecte et les références orphelines restent visibles pour revue.

In [ ]:
before = raw_profile.select("table", "column", F.col("rows").alias("raw_rows"),
                            F.col("missing").alias("raw_missing"))
after = missing.filter(F.col("stage") == "clean").select(
    "table", "column", F.col("rows").alias("clean_rows"), F.col("missing").alias("clean_missing")
)
comparison = before.join(after, ["table", "column"])
comparison.orderBy("table", "column").show(100, truncate=False)

removed = anomalies.filter(F.col("issue").isin(
    "missing_or_invalid_key", "duplicate_rows", "conflicting_rows"
)).groupBy("table").agg(F.sum("count").alias("removed"))
row_check = comparison.select("table", "raw_rows", "clean_rows").distinct().join(removed, "table")
assert row_check.filter(F.col("raw_rows") - F.col("clean_rows") != F.col("removed")).count() == 0
assert missing.filter((F.col("missing") < 0) | (F.col("missing") > F.col("rows"))).count() == 0
row_check.orderBy("table").show(truncate=False)

In [ ]:
spark.stop()